In [1]:
!pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="23foiGjwOXW66VqQrCL6")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
from google.colab import files

In [3]:
import zipfile

zip_path = "/content/aed_n_naed.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [4]:
import os
os.listdir(extract_path)

['README.dataset.txt',
 'valid',
 'test',
 'README.roboflow.txt',
 'data.yaml',
 'train']

In [5]:
import requests

API_KEY = "23foiGjwOXW66VqQrCL6"

url = f"https://api.roboflow.com/?api_key={API_KEY}"
res = requests.get(url)

print(res.json())

{'welcome': 'Welcome to the Roboflow API.', 'instructions': 'You are successfully authenticated.', 'docs': 'https://docs.roboflow.com', 'workspace': 'mosquitos-4tgsz'}


In [22]:
import os
import time

# Configuraciones del proyecto
DATASET_PATH = "/content/dataset"
PROJECT_ID = "aegypti_n_notaegypti"

project = None
try:
    project = rf.workspace().project(PROJECT_ID)
    print(f"Conectado exitosamente al proyecto: {PROJECT_ID}")
except Exception as e:
    print(f"Error al conectar con el proyecto: {e}")

def upload_with_retry(img_path, lbl_path, split, retries=3):
    for i in range(retries):
        try:
            project.upload(
                image_path=img_path,
                annotation_path=lbl_path,
                split=split
            )
            return True
        except Exception as e:
            if i < retries - 1:
                print(f"⚠️ Error subiendo {os.path.basename(img_path)} (Intento {i+1}/{retries}). Reintentando en 2s...")
                time.sleep(2)
            else:
                print(f"❌ Falló la subida de {os.path.basename(img_path)} tras {retries} intentos: {e}")
                return False

def upload_split(split):
    if project is None:
        print("No hay una instancia de proyecto válida.")
        return

    img_dir = os.path.join(DATASET_PATH, split, "images")
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")

    if not os.path.exists(img_dir):
        print(f"Saltando {split} (no hay carpeta de imágenes)")
        return

    print(f"\nSubiendo split: {split}...")
    files = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    for file in files:
        img_path = os.path.join(img_dir, file)
        lbl_path = os.path.join(lbl_dir, file.rsplit(".", 1)[0] + ".txt")

        if os.path.exists(lbl_path):
            upload_with_retry(img_path, lbl_path, split)
            time.sleep(0.5)  # Pequeña pausa para estabilidad
        else:
            print(f"⚠️ Falta etiqueta para {file}, saltando...")

if project:
    for split in ["train", "valid", "test"]:
        upload_split(split)
    print("\nProceso de subida finalizado.")
else:
    print("Abortando: El proyecto no pudo ser inicializado.")

loading Roboflow workspace...
loading Roboflow project...
Conectado exitosamente al proyecto: aegypti_n_notaegypti

Subiendo split: train...

Subiendo split: valid...

Subiendo split: test...

Proceso de subida finalizado.


In [18]:
import requests

# Corregimos la URL para pasar la API_KEY como un parámetro de consulta válido
# La estructura correcta es https://api.roboflow.com/?api_key=TU_LLAVE
url = f"https://api.roboflow.com/?api_key={rf.api_key}"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    workspace_id = data['workspace']
    print(f"Workspace detectado: {workspace_id}")

    # Ahora consultamos los detalles del workspace para listar los proyectos
    projects_url = f"https://api.roboflow.com/{workspace_id}?api_key={rf.api_key}"
    proj_res = requests.get(projects_url)

    if proj_res.status_code == 200:
        proj_data = proj_res.json()
        print("\nProyectos encontrados en tu panel:")
        for proj in proj_data['workspace']['projects']:
            print(f"- Nombre: {proj['name']} | ID: {proj['id']}")
    else:
        print(f"Error al obtener proyectos: {proj_res.status_code}")
else:
    print(f"Error al consultar la raíz de la API: {response.status_code}")
    print(response.text)

Workspace detectado: mosquitos-4tgsz

Proyectos encontrados en tu panel:
- Nombre: Aegypti_n_NotAegypti | ID: mosquitos-4tgsz/aegypti_n_notaegypti
- Nombre: ai challenge | ID: mosquitos-4tgsz/ai-challenge-y9qrl-plza5
- Nombre: Male_abd_female | ID: mosquitos-4tgsz/male_abd_female
- Nombre: Mos_and_NonMos | ID: mosquitos-4tgsz/mos_and_nonmos
- Nombre: Mos_and_NonMos 2 | ID: mosquitos-4tgsz/mos_and_nonmos-2
- Nombre: Mosquitos | ID: mosquitos-4tgsz/mosquitos-85thm
- Nombre: mosquito-and-non-mosquito | ID: mosquitos-4tgsz/
- Nombre: mosquito-and-non-mosquito | ID: mosquitos-4tgsz/
- Nombre: mosquito-and-non-mosquito | ID: mosquitos-4tgsz/
- Nombre: mosquito-and-non-mosquito | ID: mosquitos-4tgsz/
- Nombre: mosquito-5wsj2 | ID: mosquitos-4tgsz/
- Nombre: mosquito-5wsj2 | ID: mosquitos-4tgsz/
- Nombre: mosquito-5wsj2 | ID: mosquitos-4tgsz/
- Nombre: mosquito-and-non-mosquito | ID: mosquitos-4tgsz/
- Nombre: mosquito-5wsj2 | ID: mosquitos-4tgsz/
- Nombre: mosquito-5wsj2 | ID: mosquitos-4tgsz